# 情報論A 第4回

逆行列と画像の幾何変換「パノラマ画像を作ってみよう」

###準備

`samples`に入っている`pano_ref.jpg`と`pano_src.jpg`をcolabに投げ込んでください

（あるいは、Googleドライブを設定している場合、次のセルでGoogleドライブをマウントしてください）

In [ ]:
import cv2
import numpy as np  # PythonのOpenCVでは、画像はnumpyのarrayとして管理される
from google.colab.patches import cv2_imshow # colab内で画像表示関数がうまく動かないので、パッチが提供されている

# Googleドライブをマウントする場合
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/My Drive/Colab Notebooks/johoronA/"

In [ ]:
# imgをrefに張り合わせることを考える
ref = cv2.imread("pano_ref.jpg") # ベースとなる画像（BGR）
src = cv2.imread("pano_src.jpg") # 変換する画像（BGR）

# 対応点
dst_points = np.float32([[923,156],[1281,143],[1276,760],[916,745]]) # refの点(x', y')
src_points = np.float32([[88,163],[438,190],[437,760],[78,782]])  # srcの点(x, y)


##課題：連立方程式を解いてみよう

### 流れのおさらい

4組の対応点

$$
\{(x_i, y_i), (x'_i, y'_i)\}, \quad i=1,2,3,4
$$

について、未知ベクトルを

$$
\boldsymbol{h} =
\begin{bmatrix}
h_{11} \\
h_{12} \\
h_{13} \\
h_{21} \\
h_{22} \\
h_{23} \\
h_{31} \\
h_{32}
\end{bmatrix}
$$

とおく。

各対応点から、次の2本の式が得られる。

$$
x'_i
=
h_{11}x_i + h_{12}y_i + h_{13}
- h_{31}x_i x'_i
- h_{32}y_i x'_i
$$

$$
y'_i
=
h_{21}x_i + h_{22}y_i + h_{23}
- h_{31}x_i y'_i
- h_{32}y_i y'_i
$$

したがって、4組の対応点から次の連立方程式を作る。

$$
\boldsymbol{Ah}=\boldsymbol{b}
$$

ここで、

$$
\boldsymbol{A} =
\begin{bmatrix}
x_1 & y_1 & 1 & 0 & 0 & 0 & -x_1x'_1 & -y_1x'_1 \\
0 & 0 & 0 & x_1 & y_1 & 1 & -x_1y'_1 & -y_1y'_1 \\
x_2 & y_2 & 1 & 0 & 0 & 0 & -x_2x'_2 & -y_2x'_2 \\
0 & 0 & 0 & x_2 & y_2 & 1 & -x_2y'_2 & -y_2y'_2 \\
x_3 & y_3 & 1 & 0 & 0 & 0 & -x_3x'_3 & -y_3x'_3 \\
0 & 0 & 0 & x_3 & y_3 & 1 & -x_3y'_3 & -y_3y'_3 \\
x_4 & y_4 & 1 & 0 & 0 & 0 & -x_4x'_4 & -y_4x'_4 \\
0 & 0 & 0 & x_4 & y_4 & 1 & -x_4y'_4 & -y_4y'_4
\end{bmatrix}
$$

$$
\boldsymbol{b} =
\begin{bmatrix}
x'_1 \\
y'_1 \\
x'_2 \\
y'_2 \\
x'_3 \\
y'_3 \\
x'_4 \\
y'_4
\end{bmatrix}
$$

である。

4組の対応点だけを使う場合、$\boldsymbol{A}$ は $8 \times 8$ の正方行列になる。

そのため、逆行列を用いて

$$
\boldsymbol{h} = \boldsymbol{A}^{-1}\boldsymbol{b}
$$

として解くことができる。

最後に、$h_{33}=1$ としてホモグラフィ行列

$$
H =
\begin{bmatrix}
h_{11} & h_{12} & h_{13} \\
h_{21} & h_{22} & h_{23} \\
h_{31} & h_{32} & 1
\end{bmatrix}
$$

を得る。

In [ ]:
# ?の部分を埋めて、完成させましょう
# ヒント：numpyでの逆行列の計算： A_inv = np.linalg.inv(A)

A = []
b = []

for (x, y), (xp, yp) in zip(src_points, dst_points):
    # x' に関する式
    A.append([?, ?, ?, ?, ?, ?, ?, ?])
    b.append(?)

    # y' に関する式
    A.append([?, ?, ?, ?, ?, ?, ?, ?])
    b.append(?)

A = np.array(A, dtype=float)
b = np.array(b, dtype=float)

# 逆行列を使って Ah = b を解く
h = ?

print("h =")
print(h)

# h33 = 1.0 としてホモグラフィ行列 H を作る
H = np.array([
    [h[0], h[1], h[2]],
    [h[3], h[4], h[5]],
    [h[6], h[7], 1.0]
])

print("H =")
print(H)

In [ ]:
##検算（数値誤差を除き）一致するはず
H_opencv = cv2.getPerspectiveTransform(pts_src,pts_ref)  # OpenCVによるホモグラフィ行列の推定（img -> refへの変換）
print(H_opencv)

In [ ]:
# 逆変換によるパノラマスティッチング（変更不要）
## きれいな幾何変換のためには、逆変換する必要があります。詳しくはスライドの補足資料か、画像処理の教科書を読んでみてください。
## 参考：「CG-ARTS ディジタル画像処理 [改訂第二版]」8.3節など

dst = np.zeros((src.shape[0],src.shape[1]*2,3), dtype=np.uint8) # 横幅2倍の画像を生成(縦src.shape[0],横src.shape[1],3ch)
dst[0:ref.shape[0], 0:ref.shape[1], :] = ref # 最初に、refの画素値を先に入れておく（部分配列の操作）。3次元目は色なので、そのまま(:)
H_inv = np.linalg.inv(H)  # 逆変換にしたいので、逆行列を求める

for dst_y in range(dst.shape[0]):
  for dst_x in range(dst.shape[1]):
    dst_xyw = np.float32([dst_x, dst_y, 1]) # 同次座標
    src_xyw = H_inv.dot(dst_xyw)  # 変換
    src_x = src_xyw[0]/src_xyw[2] # 出力画像のX
    src_y = src_xyw[1]/src_xyw[2] # 出力画像のY

    if src_x < 1 or src_y < 1 or src_x > src.shape[1]-2 or src_y > src.shape[0]-2:  # 画像の外側を参照しないようにする
      continue

    dst[dst_y][dst_x] = src[int(src_y+0.5)][int(src_x+0.5)] # 最近傍法による補間


cv2_imshow(dst) # 表示

In [ ]:
# ↑のセルでは勉強のため自力実装しましたが、画像を変形する部分は既存の関数を使ったほうが早いです

# Appendix: OpenCVにおけるワーピング関数（デフォルトでは逆変換の後バイリニア補間）
# https://docs.opencv.org/4.5.3/da/d54/group__imgproc__transform.html#gaf73673a7e8e18ec6963e3774e6a94b87
dst_opencv = cv2.warpPerspective(src, H, (src.shape[1]*2,src.shape[0]))

cv2_imshow(dst_opencv) # 表示